In [1]:
!pip install peft

In [2]:
!pip install -U transformers trl accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
  Attempting uninstall: transformers

In [3]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.1/464.1 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.27.1
    Uninstalling huggingface-hub-0.27.1:
      Successfully uninstalled huggingface-hub-0.27.1


In [4]:
import os
import torch
from trl import SFTTrainer
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline, logging)


In [5]:
model_identifier="microsoft/Phi-3-mini-4k-instruct"

enable_4bit=True
compute_dtype_bnb="float16"
quant_type_bnb="nf4"
double_quant_flag=False

dtype_computation = getattr(torch, compute_dtype_bnb)
bnb_setup = BitsAndBytesConfig(load_in_4bit = enable_4bit,
                               bnb_4bit_quant_type = quant_type_bnb,
                               bnb_4bit_use_double_quant = double_quant_flag,
                               bnb_4bit_compute_dtype = dtype_computation)

device_assignment = {"": 0}

In [6]:
llama_model = AutoModelForCausalLM.from_pretrained(model_identifier, quantization_config = bnb_setup, device_map = device_assignment)
llama_model.config.use_case = False
llama_model.config.pretraining_tp = 1

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
llama_tokenizer = AutoTokenizer.from_pretrained(model_identifier)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"
llama_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

1

In [8]:
adapters_path = 'Saurabh2411/results'
model = PeftModel.from_pretrained(llama_model, adapters_path)

adapter_config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/403M [00:00<?, ?B/s]

In [71]:
text = "what is paracetamol poisioning ?"
device = "cuda:0"
inputs = llama_tokenizer(text, return_tensors="pt").to(device)
outputs = llama_model.generate(**inputs, max_new_tokens=500)
predicted = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(predicted)

what is paracetamol poisioning ?

A: Acute liver failure
B: Acute cholecystitis
C: Acute hepatitis
D: Acute pancreatitis


### Response:
Paracetamol (also known as acetaminophen) overdose is a common cause of acute liver failure. Paracetamol is metabolized in the liver, and in excessive amounts, it can cause hepatotoxicity, leading to liver damage. This can progress to acute liver failure, a life-threatening condition characterized by the rapid loss of liver function.

Acute cholecystitis, acute hepatitis, and acute pancreatitis are other conditions that can affect the liver, but they are not typically caused by paracetamol overdose.

Acute cholecystitis is inflammation of the gallbladder, usually due to gallstones blocking the cystic duct. Acute hepatitis is inflammation of the liver, often caused by viral infections, alcohol abuse, or autoimmune diseases. Acute pancreatitis is inflammation of the pancreas, often caused by gallstones, alcohol abuse, or certain medications.

Therefore,

In [77]:
target = "Paracetamol (also known as acetaminophen) overdose is a common cause of acute liver failure. Paracetamol is metabolized in the liver, and in excessive amounts, it can cause hepatotoxicity, leading to liver damage. This can progress to acute liver failure, a life-threatening condition characterized by the rapid loss of liver function.Acute cholecystitis, acute hepatitis, and acute pancreatitis are other conditions that can affect the liver, but they are not typically caused by paracetamol overdose."

In [78]:
import math
from collections import defaultdict
def get_n_gram_counts(words, n=4):

    n_gram_counts = { i: defaultdict(int) for i in range(1, n+1) }

    for curr_n in range(1, n+1):
        for i in range(len(words)-curr_n+1):
            current_gram = tuple(words[i:i+curr_n])
            n_gram_counts[curr_n][current_gram] += 1

    return n_gram_counts

def get_bleu_score(predicted_text, target_text):

    predicted_words, target_words = predicted_text.split(), target_text.split()

    num_predicted_words = len(predicted_words)
    num_target_words = len(target_words)

    brevity_penalty = min(1, math.exp(1 - (num_target_words/num_predicted_words)))

    predicted_n_gram_counts = get_n_gram_counts(predicted_words)
    target_n_gram_counts = get_n_gram_counts(target_words)

    total_precision = 1

    for n in range(1, 5):
        num_present = 0
        for n_gram in predicted_n_gram_counts[n]:
            num_present += min(predicted_n_gram_counts[n][n_gram], target_n_gram_counts[n][n_gram])
        precision = num_present/(num_predicted_words-n+1)
        total_precision *= precision**(0.25)

    bleu_score = brevity_penalty*total_precision

    return bleu_score

In [79]:
get_bleu_score(predicted, target)

0.24762765574612494

In [80]:
def get_rogue_score(predicted_text, target_text, n):

    predicted_words, target_words = predicted_text.split(), target_text.split()

    num_predicted_words = len(predicted_words)
    num_target_words = len(target_words)

    predicted_n_gram_counts = get_n_gram_counts(predicted_words)
    target_n_gram_counts = get_n_gram_counts(target_words)

    num_present = 0
    for n_gram in predicted_n_gram_counts[n]:
        num_present += min(predicted_n_gram_counts[n][n_gram], target_n_gram_counts[n][n_gram])
    precision = num_present/(num_predicted_words-n+1)
    recall = num_present/(num_target_words-n+1)

    f1 = 2*precision*recall/(precision + recall)

    return f1


def get_longest_common_subsequence(words1, words2, i, j):
    # can use dp
    if i == len(words1): return []
    if j == len(words2): return []
    if (words1[i] == words2[j]):
        return [words1[i]] + get_longest_common_subsequence(words1, words2, i+1, j+1)
    else:
        candidate1 = get_longest_common_subsequence(words1, words2, i+1, j)
        candidate2 = get_longest_common_subsequence(words1, words2, i, j+1)
        return candidate1 if len(candidate1) > len(candidate2) else candidate2

def get_rogue_lcs_score(predicted_text, target_text):

    predicted_words, target_words = predicted_text.split(), target_text.split()

    num_predicted_words = len(predicted_words)
    num_target_words = len(target_words)

    lcs = get_longest_common_subsequence(predicted_words, target_words, 0, 0)

    print("the longest common subsequence is: ", lcs)

    precision = len(lcs)/num_predicted_words
    recall = len(lcs)/num_target_words

    f1 = 2*precision*recall/(precision + recall)

    return f1

In [81]:
get_rogue_score(predicted, target, 1)

0.4079320113314447